In [ ]:
# margin.ipynb — 단일 월 인터랙티브 분석
# 공통 로직(원가 산정·집계·저장)은 generate_report.py 를 재사용합니다.
#   · 여러 달 일괄 생성은  →  uv run python generate_report.py
#   · 원가 = Item Code별 수량가중평균(취소전표·0원 라인 제외)
#   · 매출은 미매칭 품목도 포함(피벗과 합계 일치), 원가없는 행은 마진 N/A + Coverage(%) 표기
import importlib
import calendar, datetime
import pandas as pd
import generate_report as gr
importlib.reload(gr)

In [ ]:
# ★ 보고 월 설정
YEAR, MONTH = 2026, 6

last_day = calendar.monthrange(YEAR, MONTH)[1]
start_date = f"{YEAR}-{MONTH:02d}-01"
end_date   = f"{YEAR}-{MONTH:02d}-{last_day:02d}"
print(start_date, "~", end_date)

In [ ]:
# 원본 자료 로딩
df_s_all = gr.load_sales()          # 판매(6.2.1 Delivery Status)
cost     = gr.load_cost_table()     # Item Code별 수량가중평균 구매단가
cost.head()

In [ ]:
# 한 달치 집계 (df=Row_Data, 이하 4개 집계표)
df, df_cust, df_prod, df_item, df_bran = gr.build_month(df_s_all, cost, start_date, end_date)

# 브랜드별 — TOTAL 매출은 원본 피벗과 일치, Coverage(%)=마진산출 커버율
df_bran

In [ ]:
# 상품별 (Type 3 → Type 2)
df_prod

In [ ]:
# 업체별
df_cust

In [ ]:
# 아이템별 (Description)
df_item

In [ ]:
# Excel + PDF 저장 (파일명: sales analysis report YYYY-MM)
month_name   = datetime.date(YEAR, MONTH, 1).strftime("%b")
current_date = datetime.datetime.now().strftime("%b-%d, %Y")
stem = f"sales analysis report {YEAR}-{MONTH:02d}"

gr.save_excel(f"{stem}.xlsx", df, df_cust, df_prod, df_item, df_bran)
gr.save_pdf(f"{stem}.pdf", month_name, str(YEAR), current_date, df_cust, df_prod, df_item, df_bran)
print(f"생성 완료: {stem}.xlsx / {stem}.pdf")